In [ ]:
# 进口

import os
import random
import gradio as gr
from google import genai
from openai import OpenAI
from google.genai import types
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# 检查钥匙

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ. please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

# Gemma4 API 调用

In [ ]:
# 初始化本机 Google 客户端
# 它会自动获取 GEMINI_API_KEY 环境变量
client = genai.Client()

In [ ]:
def call_gemma(s_message, message):
    """系统系统指令""""""
    # 1. 用户留言
    messages = [
       types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]    
        )
    ]

    # 2.放置系统规则
    config = types.GenerateContentConfig(
        system_instruction=s_message,
    )
    # 3.使用原生客户端调用模型
    response = client.models.generate_content(
        model='gemma-4-31b-it', # Swap out with your preferred Gemma 4 variant
        contents=messages,
        config=config
    )

    return response.text

In [ ]:
# call_gemma(system_message, '你好')

In [ ]:
# Gemma 工具的描述和函数声明

ticket_price_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_ticket_price",
            description="Get the airline ticket price for a city",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "destination_city": {
                        "type": "STRING",
                        "description": "Destination city name"
                    }
                },
                "required": ["destination_city"]
            }
        )
    ]
)

# 不仅仅是一个城市并添加或更新票价
1. 可以同时多询问一个城市。
2.可以添加或更新城市和价格。
3. LLM仅在聊天功能中找到AVAILABLE_FUNCTIONS中的功能寄存器。功能很容易添加到上面
4. 聊天功能遵循 Google GenAI SDK 模式的规则，不需要 FunctionDeclaration，因为 SDK 会自动推断并创建模式。

In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_prices(destination_cities: list[str]):
    """返回所请求城市的机票价格。
    声明函数参数的类型以避免错误"""
    if isinstance(destination_cities, str):
        destination_cities = [destination_cities]
        
    return {
        "prices": [
            {
                "city": city,
                "price": ticket_prices.get(
                    city.lower(),
                    "Unknown ticket price"
                )
            }
            for city in destination_cities
        ]
    }

In [ ]:
system_message = """
You are a travel assistant.

Use get_ticket_prices when the user asks for ticket prices.

Use set_price_ticket when the user asks to:
- create a ticket price
- update a ticket price
- change a ticket price

Always pass city names in lowercase.
"""

def set_price_ticket(city: str, price: float):
    """创建或更新城市门票价格。"""

    city = city.lower().strip()

    existed = city in ticket_prices

    ticket_prices[city] = f"${price}"

    return {
        "success": True,
        "action": "updated" if existed else "created",
        "city": city,
        "price": f"${price}",
        "all_prices": ticket_prices
    }

In [ ]:
def chat(message, history):
    # 将历史记录转换为 GenAI 格式
    contents = []

    for item in history:
        role = item["role"]

        # 跳过系统消息，因为 Gemma 需要它们
        # 在系统指令中
        if role == "system":
            continue

        # 将 OpenAI 角色映射到 GenAI 角色
        if role == "assistant":
            genai_role = "model"
        else:
            genai_role = "user"

        contents.append(
            types.Content(
                role=genai_role,
                parts=[types.Part.from_text(text=item["content"])]
            )
        )

    # 添加当前用户消息
    contents.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    )

    relevant_system_message = system_message
    if 'belt' in message.lower(): # example
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    config = types.GenerateContentConfig(
        system_instruction=system_message,
        tools=[
            get_ticket_prices,
            set_price_ticket
            ]
    )

    # 第一次打电话给杰玛
    response = client.models.generate_content(
        model="gemma-4-31b-it",
        contents=contents,
        config=config,
    )

    # 检查函数调用
    AVAILABLE_FUNCTIONS = {
        "get_ticket_prices": get_ticket_prices,
        "set_price_ticket": set_price_ticket,
    }

    if response.candidates:
        for part in response.candidates[0].content.parts:

            if not part.function_call:
                continue

            function_name = part.function_call.name

            function_to_call = AVAILABLE_FUNCTIONS.get(function_name)

            if function_to_call is None:
                return f"Unknown tool requested: {function_name}"

            tool_result = function_to_call(
                    **dict(part.function_call.args)
                )

            contents.append(response.candidates[0].content)

            contents.append(
                types.Content(
                    role="tool",
                    parts=[
                        types.Part.from_function_response(
                            name=function_name,
                            response=tool_result
                        )
                    ]
                )
            )

            final_response = client.models.generate_content(
                model="gemma-4-31b-it",
                contents=contents,
                config=config,
            )

            return final_response.text

    return response.text

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()